# Text Processing

## Capturing Text Data

### Plain Text

In [ ]:
import os

# Read in a plain text file
with open(os.path.join("data", "hieroglyph.txt"), "r") as f:
    text = f.read()
    print(text)

### Tabular Data

In [ ]:
import pandas as pd

# Extract text column from a dataframe
df = pd.read_csv(os.path.join("data", "news.csv"))
df.head()[['publisher', 'title']]

# Convert text column to lowercase
df['title'] = df['title'].str.lower()
df.head()[['publisher', 'title']]

### Online Resource

In [ ]:
import requests
import json

# Fetch data from a REST API (note- in production avoid setting verify to False)
r = requests.get(
    "https://gist.githubusercontent.com/nmagee/1bece00a989f95411af423c940cac883/raw/6e8a16184de34e6a6e77d334a40669cc3edce8f0/qod.json", 
    verify=False) 
res = r.json()
print(json.dumps(res, indent=4))

# Extract relevant object and field
q = res["contents"]["quotes"][0]
print(q["quote"], "\n--", q["author"])

## Cleaning

The Udacity Catalog page is a JavaScript-enhanced, partially client-side rendered application. Although the catalog items do appear in the server-rendered HTML, the live page **cannot be retrieved with simple HTTP tools such as requests,** because the site is protected by a **browser verification and bot-mitigation system.** As a result, automated HTTP clients receive only a verification challenge instead of the actual catalog content.

To ensure stable access to the real data, I opened the catalog page in a real browser, allowed it to fully render, and manually saved the resulting HTML snapshot (`udacity-catalog.html`). In this notebook, we load that saved snapshot — rather than contacting the live website — to obtain reliable, complete, and reproducible HTML for parsing.

In [ ]:
# Read from a local HTML file instead of fetching from the web

file_path = "udacity-catalog.html"

with open(file_path, "r", encoding="utf-8") as f:
    html_text = f.read()

print(html_text)

Before extracting structured data, it is helpful to confirm that the snapshot contains the visible text we expect.  This step performs lightweight cleaning:

- removing `<script>` and `<style>` blocks,  
- converting common block-level tags into newline breaks,  
- stripping all remaining HTML tags,  
- collapsing excessive whitespace.

While this method does not preserve page structure, it **does reveal the raw text** present in the HTML snapshot — including lesson titles, descriptions, and navigation labels. Because Udacity performs partial server-side rendering, some program titles appear directly in the HTML as text. However, this text is unstructured and incomplete, making it unsuitable for extraction of reliable catalog metadata.

In [ ]:
import re

# Remove script and style blocks entirely
t = re.sub(r'<(script|style).*?>.*?</\1>', '', html_text, flags=re.S | re.I)

# Replace common block-level tags with newlines (preserves basic text structure)
t = re.sub(r'</?(br|p|div|li|h[1-6]).*?>', '\n', t, flags=re.I)

# Remove all remaining HTML tags
t = re.sub(r'<[^>]+>', '', t)

# Collapse multiple blank lines into a single blank line
t = re.sub(r'\n\s*\n+', '\n\n', t)

# Trim leading/trailing whitespace and print the cleaned text
print(t.strip())

BeautifulSoup provides a more structured way to examine the HTML snapshot:

- the snapshot is parsed into a DOM-like structure,  
- non-content elements such as `<style>` and `<noscript>` are removed,  
- remaining text content is extracted with line breaks preserved.

This step offers a clearer view of the page's **visible** text. Again, we observe that while program titles appear as plain text, the HTML does not include structured catalog data such as difficulty level, duration, slug, or program type. This confirms that HTML-based scraping cannot be used to reliably extract catalog metadata.

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML document
soup = BeautifulSoup(html_text, "html.parser")

# Remove style, and noscript elements (unwanted text noise)
for tag in soup(["style", "noscript"]):
    tag.decompose()
    
# Extract text with line breaks preserved
text = soup.get_text("\n")

# Strip whitespace and remove empty lines
clean = "\n".join(line.strip() for line in text.splitlines() if line.strip())

# Output cleaned text
print(clean)

All structured catalog data is stored inside a special script tag created by Next.js:

```html
<script id="__NEXT_DATA__"> … JSON … </script>
```

This JSON object contains the full application state consumed by the React frontend. It includes program titles, summaries, metadata fields, pagination info, and filtering categories — none of which exist in structured form within the HTML.

Extracting and parsing this JSON payload is therefore the **authoritative** way to access catalog information.

In [ ]:
import json

# Find the JSON blob that contains catalog data
next_data = soup.find("script", id="__NEXT_DATA__")

# Parse JSON
data = json.loads(next_data.string)

# Extract lesson titles from catalogSearchResults
items = data["props"]["pageProps"]["catalogSearchResults"]["searchResultItems"]
titles = [item["title"] for item in items]

# \Print all lesson titles
for t in titles:
    print(t)

To confirm that the parsed JSON structure behaves as expected, we access the first item in

```
props → pageProps → catalogSearchResults → searchResultItems
```

and print its `"title"` value. This step validates that:

- the JSON was parsed correctly,
- the expected data path exists,
- and catalog items are accessible as structured Python objects.

In [ ]:
# Extract the title of the first lesson
first_title = items[0]["title"]

print(first_title)

Finally, we iterate over all catalog entries and collect the `"title"` field into a clean Python list. While the HTML snapshot contains text versions of these titles, the JSON-derived list ensures:

- the titles are complete and up to date,  
- the order matches the live catalog’s logic,  
- and the extraction includes only catalog items (not navigation, footer text, or marketing text).

This structured list can now be used for further analysis, exporting, or other processing tasks.

In [ ]:
# Convert to original output structure
lessons = []
for item in items:
    title = item["title"].strip()
    lessons.append(title)

print(f"{len(lessons)} lessons found. Sample: '{lessons[0]}'")

## Normalization

### Case Normalization

In [ ]:
# Sample text
text = "The first time you see The Second Renaissance it may look boring. Look at it at least twice and definitely watch part 2. It will change your view of the matrix. Are the human people the ones who started the war ? Is AI a bad thing ?"
print(text)

In [ ]:
# Convert to lowercase
text = text.lower() 
print(text)

### Punctuation Removal

In [ ]:
import re

# Remove punctuation characters
text = re.sub(r"[^a-zA-Z0-9]", " ", text) 
print(text)

## Tokenization

In [ ]:
# Split text into tokens (words)
words = text.split()
print(words)

### NLTK: Natural Language ToolKit

In [ ]:
import os
import nltk

nltk.data.path.append(os.path.join(os.getcwd(), "nltk_data"))

In [ ]:
# Another sample text
text = "Dr. Smith graduated from the University of Washington. He later started an analytics firm called Lux, which catered to enterprise customers."
print(text)

In [ ]:
from nltk.tokenize import word_tokenize

# Split text into words using NLTK
words = word_tokenize(text)
print(words)

In [ ]:
from nltk.tokenize import sent_tokenize

# Split text into sentences
sentences = sent_tokenize(text)
print(sentences)

In [ ]:
# List stop words
from nltk.corpus import stopwords
print(stopwords.words("english"))

In [ ]:
# Reset text
text = "The first time you see The Second Renaissance it may look boring. Look at it at least twice and definitely watch part 2. It will change your view of the matrix. Are the human people the ones who started the war ? Is AI a bad thing ?"

# Normalize it
text = re.sub(r"[^a-zA-Z0-9]", " ", text.lower())

# Tokenize it
words = text.split()
print(words)

In [ ]:
# Remove stop words
words = [w for w in words if w not in stopwords.words("english")]
print(words)

## Part of Speech Tagging/ Sentence Parsing

In [ ]:
import nltk

# Define a custom grammar
my_grammar = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")
parser = nltk.ChartParser(my_grammar)

# Parse a sentence
sentence = word_tokenize("I shot an elephant in my pajamas")
for tree in parser.parse(sentence):
    print(tree)

## Stemming & Lemmatization

### Stemming

In [ ]:
from nltk.stem.porter import PorterStemmer

# Reduce words to their stems
stemmed = [PorterStemmer().stem(w) for w in words]
print(stemmed)

### Lemmatization

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer

# Reduce words to their root form
lemmed = [WordNetLemmatizer().lemmatize(w) for w in words]
print(lemmed)

In [ ]:
# Lemmatize verbs by specifying pos
lemmed = [WordNetLemmatizer().lemmatize(w, pos='v') for w in lemmed]
print(lemmed)